# Gene Panel Compression — Exploration Notebook

Interactive walkthrough of the full pipeline:
1. Load a region from the ABC Atlas
2. Run baseline selections (PCA, DE, HVG, random, Spapros)
3. Train a Concrete Autoencoder end-to-end
4. Compare all methods across panel sizes
5. Visualize: UMAP, reconstruction scatter, spatial maps


In [ ]:
import sys, os
sys.path.insert(0, '..')   # experiments/gene_compression/
sys.path.insert(0, '../../..')  # spapros repo root (for local spapros install)

import numpy as np
import pandas as pd
import scanpy as sc
import torch
import matplotlib.pyplot as plt

sc.settings.verbosity = 1
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

## 1. Load ABC Atlas data

In [ ]:
from src.data.abc_atlas import load_abc_atlas, make_loaders

# On Code Ocean: data is at /data/abc_atlas/
# Locally: set ABC_ATLAS_PATH env var or pass data_root directly
DATA_ROOT = os.environ.get('ABC_ATLAS_PATH', '/data/abc_atlas')

adata = load_abc_atlas(
    region='WMB-10Xv3-TH',   # Thalamus
    data_root=DATA_ROOT,
    celltype_key='subclass',
    n_hvg=3000,
    max_cells=50_000,          # subsample for quick exploration
)
print(adata)

In [ ]:
# Quick overview of cell type distribution
ct_counts = adata.obs['subclass'].value_counts()
ct_counts.head(20).plot(kind='barh', figsize=(8, 6))
plt.xlabel('Cell count')
plt.title('Cell type distribution (subclass)')
plt.tight_layout()

## 2. Baseline gene selections

In [ ]:
from src.baselines.gene_selection import select_pca, select_de, select_hvg, select_random

PANEL_SIZE = 50   # change to 10, 25, 100

genes_pca    = select_pca(adata, PANEL_SIZE)
genes_de     = select_de(adata, PANEL_SIZE, celltype_key='subclass')
genes_hvg    = select_hvg(adata, PANEL_SIZE)
genes_random = select_random(adata, PANEL_SIZE, seed=42)

print(f'PCA    : {genes_pca[:5]} ...')
print(f'DE     : {genes_de[:5]} ...')
print(f'HVG    : {genes_hvg[:5]} ...')
print(f'Random : {genes_random[:5]} ...')

## 3. Train Concrete Autoencoder

In [ ]:
from src.models.autoencoder import build_model
from src.training.losses import CompressionLoss
from src.training.trainer import Trainer, TemperatureScheduler

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using: {device}')

n_genes = int(adata.var['highly_variable'].sum())
n_celltypes = len(adata.uns['celltypes'])

model = build_model({
    'n_genes': n_genes,
    'n_selected': PANEL_SIZE,
    'n_celltypes': n_celltypes,
    'latent_dim': 64,
    'encoder_dims': [256, 128],
    'decoder_dims': [128, 256],
    'selector_type': 'concrete',
    'temperature': 1.0,
})
print(model)

In [ ]:
train_loader, val_loader, test_loader = make_loaders(
    adata, batch_size=512, num_workers=0   # num_workers=0 for notebooks
)

loss_fn = CompressionLoss(lambda_recon=1.0, lambda_ct=0.5)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
lr_sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
temp_sched = TemperatureScheduler(t_start=1.0, t_min=0.01, anneal_rate=3e-4)

trainer = Trainer(
    model, loss_fn, optimizer,
    scheduler=lr_sched,
    temp_scheduler=temp_sched,
    device=device,
    save_dir='../../../results/notebooks/concrete_ae',
    log_every=200,
)

history = trainer.fit(train_loader, val_loader, n_epochs=50, early_stop_patience=10)

In [ ]:
from src.visualization.plots import plot_training_history
fig = plot_training_history(history)
plt.show()

In [ ]:
# What genes did the model learn to select?
hvg_names = adata.var_names[adata.var['highly_variable']].tolist()
selected_genes_ae = model.get_selected_genes(var_names=hvg_names)
print(f'Concrete AE selected {len(selected_genes_ae)} genes:')
print(selected_genes_ae)

## 4. Evaluate and compare all panels

In [ ]:
from src.evaluation.metrics import compare_panels

panels = {
    'concrete_ae': selected_genes_ae,
    'pca':         genes_pca,
    'de':          genes_de,
    'hvg':         genes_hvg,
    'random':      genes_random,
}

results = compare_panels(
    adata, panels,
    celltype_key='ct_label',
    run_clustering=True,
    run_knn=True,
    run_ct_f1=True,
)
results

## 5. Visualize

In [ ]:
# Get latent embeddings for UMAP
model.eval()
from src.data.abc_atlas import AnnDataset
dataset = AnnDataset(adata)
loader = torch.utils.data.DataLoader(dataset, batch_size=1024, shuffle=False)

all_h = []
with torch.no_grad():
    for batch in loader:
        x = batch['x'].to(device)
        out = model(x)
        all_h.append(out['h'].cpu().numpy())
embeddings = np.vstack(all_h)
print('Latent shape:', embeddings.shape)

In [ ]:
from src.visualization.plots import plot_umap_comparison

genes_in = [g for g in selected_genes_ae if g in adata.var_names]
adata_panel = adata[:, genes_in].copy()
adata_full  = adata[:, adata.var['highly_variable']].copy()

fig = plot_umap_comparison(
    adata_full, adata_panel, embeddings,
    celltype_key='subclass',
)
plt.show()

In [ ]:
# Reconstruction quality scatter
from src.visualization.plots import plot_reconstruction_scatter
import scipy.sparse as sp

X_orig = adata_full.X
if sp.issparse(X_orig):
    X_orig = X_orig.toarray()

# Get reconstructions
model.eval()
all_xhat = []
with torch.no_grad():
    for batch in loader:
        x = batch['x'].to(device)
        out = model(x)
        all_xhat.append(out['x_hat'].cpu().numpy())
X_recon = np.vstack(all_xhat)

fig = plot_reconstruction_scatter(
    X_orig, X_recon,
    n_genes_to_show=6,
    var_names=hvg_names,
)
plt.show()